In [2]:
using LinearAlgebra

"""
计算 Chebyshev-Gauss-Lobatto 点及其积分权重
输入: N (多项式阶数，点的个数为 N+1)
输出: x (配置点), w (权重向量)
"""
function cheb_quad(N::Int)
    # 1. 生成 Chebyshev-Gauss-Lobatto 配置点
    # x_j = cos(j * π / N), j = 0, ..., N
    x = [cos(pi * j / N) for j in 0:N]
    
    # 2. 初始化权重向量
    w = zeros(N + 1)
    
    # 3. 直接构造权重 (Clenshaw-Curtis)
    # 参考 Waldvogel (2006) 的矩阵化表述，不使用 FFT
    n = 0:N
    for j in 0:N
        # 计算每个点 x_j 对应的权重
        # s 是针对中间点的求和项
        s = 0.0
        # 内部求和只针对偶数项 k = 2, 4, ..., 2*floor(N/2)
        for k in 1:floor(Int, N/2)
            term = 2.0 / (1.0 - (2*k)^2) * cos(2 * k * j * pi / N)
            # 边界修正：如果 2k == N，该项权重减半
            if 2*k == N
                s += 0.5 * term
            else
                s += term
            end
        end
        
        # 组合各项
        # w_j = (1 + sum) * (c_j / N)，其中 c_j 在边界点为 1，中间点为 2
        c_j = (j == 0 || j == N) ? 1.0 : 2.0
        w[j+1] = (c_j / N) * (1.0 + s)
    end
    
    return x, w
end

# --- 使用示例 ---
N = 100  # 阶数
x, w = cheb_quad(N)

# 验证：积分函数 f(x) = 1，区间 [-1, 1] 结果应为 2
integral_1 = sum(w)
println("Integral of f(x)=1: ", integral_1)

# 验证：积分函数 f(x) = x^2，结果应为 2/3
f = x.^2
integral_x2 = dot(w, f)
println("Integral of f(x)=x^2: ", integral_x2)

Integral of f(x)=1: 2.0
Integral of f(x)=x^2: 0.6666666666666667


In [5]:
reverse(w)

101-element Vector{Float64}:
 0.00010001000100010482
 0.0009634278641084526
 0.0019808179271982264
 0.0029524560961472226
 0.0039398419035366694
 0.004912969725020091
 0.005887861820873181
 0.006852350689068432
 0.00781345860526409
 0.008764257339122198
 ⋮
 0.007813458605264072
 0.006852350689068419
 0.00588786182087318
 0.004912969725020089
 0.003939841903536667
 0.0029524560961472204
 0.0019808179271982264
 0.0009634278641084548
 0.00010001000100010482